In [ ]:
import requests, pandas as pd

BASE = "https://data.ny.gov/resource/wujg-7c2s.json"
params = {
    "$select": "transit_timestamp,station_complex,borough,ridership,latitude,longitude,georeference",
    "$where": ("upper(transit_mode) like 'SUBWAY%' and "
               "transit_timestamp between '2023-01-01T00:00:00' and '2024-12-31T23:59:59'"),
    "$order": "station_complex,transit_timestamp",
    "$limit": 50000,
    "$offset": 0
}

rows = []
while True:
    r = requests.get(BASE, params=params, timeout=60)
    r.raise_for_status()
    batch = r.json()
    rows.extend(batch)
    if len(batch) < params["$limit"]:
        break
    params["$offset"] += params["$limit"]

df = pd.DataFrame(rows)
# 类型整理（可选）
df["ridership"] = pd.to_numeric(df["ridership"], errors="coerce")
df["transit_timestamp"] = pd.to_datetime(df["transit_timestamp"])

df.to_csv("nyc_subway_hourly_2023_2024_selected_cols.csv", index=False)
print("saved", len(df))